# MyCon — Colab Pipeline (3D reconstruction + BIM + VLM)

MyCON research integration notebook — **CPU-contract-tested orchestration, not field-certified performance**.
Use a stable Run ID to resume and enable heavy stages only after supplying real inputs.

**Built for long, unattended runs on unstable Colab runtimes:**
- Stage outputs use the configured project root on Drive; fast local caches are
  periodically synchronized, but unexpected Colab shutdowns may lose unwritten data.
- Original inputs/checkpoints are never overwritten intentionally; force is **off** by default.
- **Checkpoint/resume**: progress is recorded after every stage. If Colab
  disconnects, re-attach, re-run cells 1–4 with the **same Run ID**, and the
  pipeline skips everything already finished and continues.
- **Automated setup**: system packages, Python deps, COLMAP, ffmpeg, and an
  optional real local VLM (Ollama + Qwen-VL) are installed for you.
- **Portable**: because the whole project tree (config + state + artifacts)
  lives on Drive, you can resume from another device or Drive account.

## Execution profiles
- `colab_safe` — light resource profile; availability still depends on runtime and input data.
- `colab_gpu`   — full single-GPU pipeline (real dense/cleanup); real VLM if provisioned.
- `production`  — server-grade settings (use an A100/L4 high-RAM runtime).


## 0. Runtime check (read this first)

1. **Runtime → Change runtime type → GPU (T4 or better)** before heavy stages.
2. **High-RAM** helps Stage 5/7. Every cell below is idempotent and safe to re-run.


In [ ]:
import os, sys, platform, shutil, subprocess
print('python   :', sys.version.split()[0])
print('platform :', platform.platform())
print('is_colab :', 'google.colab' in sys.modules or os.path.exists('/content'))
for tool in ('ffmpeg', 'colmap', 'git'):
    print(f'{tool:7s}: {shutil.which(tool) or "NOT FOUND (installed in step 2)"}')
try:
    print('\nGPU:\n' + subprocess.check_output(['nvidia-smi', '-L'], text=True))
except Exception as exc:
    print('\nnvidia-smi:', exc, '(no GPU — pick a GPU runtime for heavy stages)')


## 1. Clone the MyCon repository

The final research fixes live on the **Integration branch**, not `main`.
An existing clone must have the correct branch and no uncommitted changes; pulls fail closed.


In [ ]:
import os, subprocess, sys, pathlib
REPO_URL = 'https://github.com/sayyedalimrj/MyCon.git'
BRANCH   = 'research/mycon-integration-20261112'
CLONE_DIR = pathlib.Path('/content/MyCon')
if not CLONE_DIR.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(CLONE_DIR)])
else:
    if not (CLONE_DIR / '.git').is_dir():
        raise RuntimeError('Existing MyCon directory is not a Git clone; no files were removed')
    current_branch = subprocess.check_output(['git','-C',str(CLONE_DIR),'branch','--show-current'], text=True).strip()
    if current_branch != BRANCH:
        raise RuntimeError(f'Existing checkout on {current_branch!r}, expected {BRANCH!r}; refusing silent branch switching')
    dirty = subprocess.check_output(['git','-C',str(CLONE_DIR),'status','--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('Uncommitted changes detected; refusing to pull over modified local files')
    subprocess.check_call(['git','-C',str(CLONE_DIR),'pull','--ff-only','origin',BRANCH])
git_sha = subprocess.check_output(['git','-C',str(CLONE_DIR),'rev-parse','HEAD'], text=True).strip()
print('MyCON checked-out SHA:', git_sha)
REPO_ROOT = CLONE_DIR / 'repository' if (CLONE_DIR / 'repository').is_dir() else CLONE_DIR
print('REPO_ROOT =', REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)


## 2. Install dependencies (system + Python)

Installs system tools, core/DA3 and optional research-geometry/benchmark dependencies.
On a new runtime, installation may take substantial time or fail; verify status below
before launching a heavy stage. Installing packages never certifies a real reconstruction.


In [ ]:
from colab.environment import bootstrap_environment, format_validation
from colab.log_capture import LogBuffer
boot_log = LogBuffer(max_lines=6000)
summary = bootstrap_environment(REPO_ROOT, install_apt=True, install_da3=True, install_ui=True, log=boot_log)
print('environment ok:', summary.get('ok'))
if not summary.get('ok'):
    raise RuntimeError('Dependency bootstrap failed — inspect environment logs before running')
print('gpu:', summary.get('gpu'))
for r in summary.get('python', []):
    print(f"  [{ 'OK ' if r['ok'] else 'FAIL' }] { r['name'] } — { r['detail'] }")


## 3. Mount Google Drive and create / resume the project tree

All outputs live under `MyDrive/MyCon_Colab/projects/<RUN_ID>/`. Re-use the same
`RUN_ID` to resume a previous run. A background daemon mirrors fast local caches
(Hugging Face, models) onto Drive every couple of minutes.


In [ ]:
from colab.drive import setup_project_tree, remount_drive, DEFAULT_DRIVE_BASE
from colab.sync import DriveSyncManager
from colab.checkpoint import CheckpointManager
# Keep this ID **unchanged across Colab reconnects**; choose your own ID on first use.
RUN_ID = 'mycon_site_01'
PROFILE = 'colab_gpu'  # 'colab_safe' | 'colab_gpu' | 'production'

PROJECT = setup_project_tree(run_id=RUN_ID, drive_base=DEFAULT_DRIVE_BASE, log=boot_log)
if not PROJECT.on_drive:
    raise RuntimeError('Google Drive not mounted/writable; refusing a non-persistent project run')
for k, v in PROJECT.as_dict().items():
    print(f'  {k:20s} = {v}')

# Resilient background Drive sync for local caches.
SYNC = None
if PROJECT.on_drive:
    SYNC = DriveSyncManager(drive_mount=PROJECT.drive_mount, log=boot_log, interval=120.0)
    SYNC.set_remount_callback(lambda: remount_drive(log=boot_log))
    SYNC.register(PROJECT.local_hf_cache_dir, PROJECT.hf_cache_dir)
    SYNC.start_periodic()


## 4. Launch the Gradio UI (recommended)

The UI uses private Colab inline access by default. Public sharing is opt-in
and exposes a temporary link; only enable it if you accept the access risk. It has
tabs for *Project & Inputs*, *Run Pipeline* (with **Resume** + retries + a
**Run FULL pipeline** button), *Artifacts & Downloads*, and *Environment,
Models & Cleanup* (one-click **Provision real local VLM**).

Prefer scripting? Skip to *Section 5* for the headless API.


In [ ]:
from colab.ui import build_ui
ui = build_ui(repo_root=REPO_ROOT, log=boot_log)
ui.queue()
PUBLIC_SHARE = False  # change only when you deliberately want a public Gradio URL
_ = ui.launch(share=PUBLIC_SHARE, inline=True, prevent_thread_lock=True)


## 5. Headless / scripted execution (no UI)

The cells below do exactly what the UI does, from Python. Use them for
fully unattended runs.


In [ ]:
# 5a) Write the effective config (profile + your overrides) onto Drive.
from colab.config_manager import build_effective_config, write_effective_config, validate_effective_config
VIDEO_PATH    = None  # e.g. PROJECT.uploads_dir / 'site_walkthrough.mp4' (copy your video there first)
IFC_PATH      = None  # e.g. PROJECT.uploads_dir / 'model.ifc'
SCHEDULE_PATH = None  # e.g. PROJECT.uploads_dir / 'schedule.csv'
USER_OVERRIDES = '''
# Free-form YAML merged on top of the profile. Example:
# dense:
#   max_image_size: 800
cleanup:
  comparison_ready_enabled: true  # opt-in finite geometry analysis; non-destructive
'''
data = build_effective_config(
    repo_root=REPO_ROOT, project_root=PROJECT.project_root, run_id=PROJECT.run_id,
    project_name=PROJECT.run_id, video_path=VIDEO_PATH, ifc_path=IFC_PATH,
    schedule_path=SCHEDULE_PATH, profile=PROFILE, user_overrides_yaml=USER_OVERRIDES, log=boot_log,
)
CONFIG_PATH = write_effective_config(data=data, out_path=PROJECT.active_config_path, log=boot_log)
ok, detail = validate_effective_config(config_path=CONFIG_PATH, repo_root=REPO_ROOT)
print('config validates:', ok, '|', detail, '|', CONFIG_PATH)
if not ok:
    raise RuntimeError('Active config did not validate; refusing pipeline execution')


In [ ]:
# 5b) (Optional) Provision a REAL local VLM (Ollama + Qwen-VL), cached on Drive.
#      Stage 7.5 remains deterministic/mock-only; this enables real Stage 10 only.
#      Skip this cell if you do not want to install/pull an Ollama model.
from colab.models import provision_vlm, DEFAULT_VLM_MODEL
VLM = provision_vlm(model=DEFAULT_VLM_MODEL, models_dir=PROJECT.model_cache_dir / 'ollama', log=boot_log)
print(VLM)
if VLM.ok:
    data = build_effective_config(
        repo_root=REPO_ROOT, project_root=PROJECT.project_root, run_id=PROJECT.run_id,
        project_name=PROJECT.run_id, video_path=VIDEO_PATH, ifc_path=IFC_PATH,
        schedule_path=SCHEDULE_PATH, profile=PROFILE,
        override_dict=VLM.data['config_overrides'], user_overrides_yaml=USER_OVERRIDES, log=boot_log,
    )
    CONFIG_PATH = write_effective_config(data=data, out_path=PROJECT.active_config_path, log=boot_log)
    ok, detail = validate_effective_config(config_path=CONFIG_PATH, repo_root=REPO_ROOT)
    if not ok:
        raise RuntimeError(f'VLM settings invalid; refusing execution: {detail}')


## 5c. Run the pipeline — full, resumable, and **stage-selectable**

The default is a limited Colab-safe stage subset, **not** the full production run.
Set `RUN_FULL_PIPELINE = True` explicitly with a real video and adequate hardware.
Resume skips checkpointed outputs when their files are still available. Force is off.
Select `START_FROM` only from declared stage names; unknown names are rejected.
Per-stage retries are best-effort and Stage 8/9 require independent BIM/metric evidence.


In [ ]:
from colab.stage_runner import (run_stages, keys_from_stage,
    FULL_PIPELINE_KEYS, COLAB_SAFE_DEFAULT_KEYS)

# Options:
# The heavy reconstruction is never started by default.
RUN_FULL_PIPELINE = False  # explicitly True after uploading a real video
START_FROM = None  # e.g. 'stage_05_dense' (requires previously completed stages)
if START_FROM is not None and START_FROM not in FULL_PIPELINE_KEYS:
    raise ValueError(f'Unknown stage {START_FROM!r} — refusing accidental full run')
if RUN_FULL_PIPELINE and not (VIDEO_PATH or data.get('inputs', {}).get('video')):
    raise RuntimeError('Full pipeline needs VIDEO_PATH or configured inputs.video')
STAGES = (keys_from_stage(START_FROM) if START_FROM
          else list(FULL_PIPELINE_KEYS) if RUN_FULL_PIPELINE
          else list(COLAB_SAFE_DEFAULT_KEYS))
print('stages to run:', STAGES)

results = run_stages(
    spec_keys=STAGES, config_path=CONFIG_PATH, repo_root=REPO_ROOT,
    logs_dir=PROJECT.logs_dir, reports_dir=PROJECT.reports_dir, log=boot_log,
    project_root=PROJECT.project_root, run_id=PROJECT.run_id,
    resume=True, force=False, max_attempts=2, stop_on_failure=True,
    hf_cache_dir=PROJECT.local_hf_cache_dir, sync_manager=SYNC,
    extra_kv={'question': 'Summarize available evidence and progress.'},
)
for r in results:
    tag = 'SKIP' if r.skipped else ('OK ' if r.ok else 'FAIL')
    print(f'  [{tag}] {r.key} rc={r.return_code} {r.duration_sec:.1f}s attempts={r.attempts}')


## 6. Inspect progress, artifacts and build a download bundle


In [ ]:
import json
# Checkpoint manifest (per-stage status, survives disconnects):
state = json.loads(PROJECT.run_state_path.read_text()) if PROJECT.run_state_path.exists() else {}
for key, st in (state.get('stages') or {}).items():
    print(f"  {key:28s} {st.get('status'):10s} attempts={st.get('attempts')} {st.get('duration_sec',0):.0f}s")

from colab.artifacts import collect_artifacts, build_artifact_bundle
entries = collect_artifacts(PROJECT.project_root)
for e in entries[:60]:
    print(f'  {e.category:16s} {e.bytes/1024:8.1f} KB  {e.relative_path}')
# bundle = build_artifact_bundle(project_root=PROJECT.project_root, exports_dir=PROJECT.exports_dir,
#                                categories=['reports','exports_viewer','cleanup','dense','vlm_qa'])
# print('bundle:', bundle)


## 7. Memory & sync utilities


In [ ]:
from colab.cleanup import free_memory, disk_usage_summary
print(free_memory(verbose=True))
if SYNC is not None:
    print('sync flush:', SYNC.flush())
for path, info in disk_usage_summary([REPO_ROOT, PROJECT.project_root]).items():
    print(f'  {path} -> {info}')


## 8. Resume, stage selection & how persistence works

### How checkpointing works
- After **every stage** (and every state change) a manifest is written
  **atomically** (temp file + `os.replace`) to
  `runs/<RUN_ID>/reports/run_state.json`, so a crash never corrupts it.
- A stage counts as *complete* only if its status is `ok`/`skipped` **and** its
  declared output files still exist on Drive — a half-synced stage re-runs.

### How Drive syncing works
- `project.root` lives on Drive, so every stage writes straight to Drive.
- Fast local caches (Hugging Face, models) are mirrored to Drive by a background
  daemon **~every 60 s** and flushed at each stage boundary — progress is saved
  continuously during a run, not only at the end.
- All writes use temp-file + `os.replace`; a stale Drive mount is auto-remounted.

### Resume after an interruption
- Re-run cells 1–4, then **re-run cell 5c with the same `RUN_ID`**. Finished
  stages are skipped; the run continues from the first incomplete stage.

### Choose which stage to start from
- Set `START_FROM = 'stage_05_dense'` (any stage key) in cell 5c, or use the
  **Start from stage** dropdown on Tab 2 of the Gradio UI. Earlier stages are not
  re-run. It does **not** always restart from the beginning.

### Continue on another device / Drive account
- Copy or share `MyDrive/MyCon_Colab/projects/<RUN_ID>/` to the new Drive, set the
  same `RUN_ID`, and run. The config + checkpoint manifest travel with the folder.

### Caveats
- The heavy stages (COLMAP sparse/dense) are external binaries, so within those
  stages progress is preserved by the ~60 s Drive sync rather than a frame-level
  checkpoint. The **video example notebook** adds true frame-level mid-stage resume.
- **Stage 5 (dense)** can OOM on a free T4 at large `dense.max_image_size`; use the
  `colab_safe` profile or set `dense.max_image_size: 800` in USER_OVERRIDES.
- **Stage 8/9** require real metric controls/IFC evidence; a blocked metric alignment
  returns a nonzero exit code rather than pretending calibration succeeded.
- **Stage 7.5** is deterministic/mock-only; real grounded VLM inference belongs to Stage 10.
- The comparison-ready Stage 7 output has unverified physical scale until surveyed.
- An old completed Stage 7 checkpoint is not silently reprocessed: use a new Run ID
  or an explicit create-only geometry analysis run for that source cloud.
- Free Colab disconnects after ~12 h; everything is on Drive, so just resume.
